# 02 – Data integration

**Goal:** Combine the four sources into consistent tables with common keys and document every join: key, rows before and after, and the reason for every lost record.
**Input:** `data/raw/`, `data/interim/jolpica_*.parquet`, `data/interim/wikipedia_circuits.parquet` · **Output:** `data/interim/{circuits,races,drivers,driver_map,results,laps,weather}.parquet`, `reports/logs/integration_log.csv`
**Assessment items:** P2 (preparation and integration of at least two sources, documented)

| Step | Left | Right | Key | Method |
|---|---|---|---|---|
| 1 | FastF1 sessions | Jolpica calendar | season + round | exact join, then name/date check |
| 2 | Jolpica circuits | GeoJSON locations | coordinates | nearest point (haversine) ≤ 5 km |
| 2b | Jolpica circuits | Wikipedia circuit list | circuit name (+ town) | fuzzy match (rapidfuzz) + manual correction table |
| 3 | FastF1 drivers | Jolpica drivers | season + round + car number | exact join, check against FastF1 DriverId and abbreviation |
| 4 | Jolpica results | Jolpica qualifying | season + round + driverId | left join |
| 5 | FastF1 laps | driver mapping (step 3) | season + round + car number | exact join |
| 6 | FastF1 weather | laps | session time | as-of join (backward) when building pairs, notebook 04 |

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
from src.config import INTERIM_DIR, PILOT_SEASONS
from src.integrate import integrate

SEASONS = PILOT_SEASONS
jolpica = {name: pd.read_parquet(INTERIM_DIR / f"jolpica_{name}.parquet")
           for name in ("races", "results", "qualifying")}
wiki = pd.read_parquet(INTERIM_DIR / "wikipedia_circuits.parquet")

tables, log = integrate(SEASONS, jolpica, wiki)

## 1. Integration log

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(log.style.hide(axis="index").set_properties(**{"text-align": "left"}))

## 2. Circuit matching (GeoJSON and Wikipedia)

In [ ]:
tables["circuit_matching"][["circuit_id", "circuit_name", "locality", "geojson_id",
                            "geo_distance_km", "wiki_name", "wiki_match", "wiki_score",
                            "circuit_type", "length_km", "turns"]]

## 3. Data quality of the lap table

In [ ]:
laps = tables["laps"]
quality = pd.Series({
    "laps": len(laps),
    "position missing": int(laps["position"].isna().sum()),
    "lap time missing": int(laps["lap_time_s"].isna().sum()),
    "IsAccurate = False": int((~laps["is_accurate"].astype(bool)).sum()),
    "pit-in laps": int(laps["pit_in"].sum()),
    "pit-out laps": int(laps["pit_out"].sum()),
})
quality.to_frame("count")

In [ ]:
laps["compound"].value_counts(dropna=False).to_frame("laps")

## 4. Resulting tables

In [ ]:
pd.Series({name: len(df) for name, df in tables.items()
           if name != "circuit_matching"}).to_frame("rows")